# Coffee17 F0 seed 42 — Kaggle primary

Kaggle-only workflow. Tambahkan dataset **Coffee Green Bean with 17 Defects Original** sebagai Input. Tidak memakai Google Drive atau Hugging Face.

Untuk **R0 pertama**, jalankan cell setup/audit dan cell audit summary dulu; jangan jalankan training sampai output audit diperiksa.


In [ ]:
ARM='F0'
CODE_COMMIT='7de2abb46efd5e71dbab508e2ae4b4e61102a7aa'

import hashlib, importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working')
PROJECT=WORK/'coffee17-preprocessing-project'
REPO=WORK/'coffee-bean-classification-code'
assert INPUT.is_dir() and WORK.is_dir(), 'Notebook ini harus dijalankan di Kaggle.'

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def merge_tree_exact(source, target):
    source=Path(source); target=Path(target)
    for item in sorted(source.rglob('*')):
        if not item.is_file():
            continue
        rel=item.relative_to(source)
        dst=target/rel
        dst.parent.mkdir(parents=True,exist_ok=True)
        if dst.is_file():
            if sha256_file(item)!=sha256_file(dst):
                raise RuntimeError(f'Kaggle input conflict: {rel}')
        else:
            shutil.copy2(item,dst)

prior_projects=sorted(
    p for p in INPUT.rglob('coffee17-preprocessing-project')
    if p.is_dir()
)
for prior in prior_projects:
    print('MERGE PRIOR KAGGLE OUTPUT:',prior)
    merge_tree_exact(prior,PROJECT)

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git','clone','--quiet','--no-checkout',
    'https://github.com/ediprin/coffee-bean-classification.git',
    str(REPO)
],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--quiet','--detach',CODE_COMMIT],check=True)

prior_lock=PROJECT/'evidence/coffee17-preprocessing-runtime-v1/requirements_preprocessing_study_lock.txt'
requirements=prior_lock if prior_lock.is_file() else REPO/'requirements/preprocessing-study.txt'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(requirements)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src'))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Aktifkan Kaggle GPU sebelum menjalankan notebook.')
print('GPU:',torch.cuda.get_device_name(0))

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import materialize_preprocessing_development
from bilinear_lmmd.core.run_lock import exclusive_training_lock
from bilinear_lmmd.experiments.preprocessing_environment import freeze_environment, verify_environment
from bilinear_lmmd.experiments.run_preprocessing_static_preflight import run_preprocessing_static_preflight
from bilinear_lmmd.experiments.run_preprocessing_observability_audit import run_observability_audit
from bilinear_lmmd.experiments.verify_preprocessing_reference_equivalence import verify_reference_equivalence

# Gunakan Coffee17 yang dipasang sebagai Kaggle Input; tidak ada download dataset eksternal.
by_class=discover_directory_samples(INPUT)
print('Coffee17 mounted:',sum(len(v) for v in by_class.values()),'images /',len(by_class),'classes')

ARCHIVE=WORK/'coffee17_original.zip'
if ARCHIVE.exists():
    ARCHIVE.unlink()
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info=zipfile.ZipInfo(f'{class_name}/{path.name}',date_time=(1980,1,1,0,0,0))
            info.compress_type=zipfile.ZIP_STORED
            info.external_attr=0o644 << 16
            bundle.writestr(info,path.read_bytes())

PROV_LOCAL=WORK/'coffee17_provenance'
CANONICAL=WORK/'coffee17_original_v1'
FOLDS_LOCAL=WORK/'coffee17_folds'
for path in (PROV_LOCAL,CANONICAL,FOLDS_LOCAL):
    if path.exists():
        shutil.rmtree(path)

provenance=audit_coffee17_provenance(ARCHIVE,PROV_LOCAL,canonical_root=CANONICAL)
if provenance['decision']!='PASS':
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary=prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL/'coffee17_provenance.json',
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary['decision']!='PASS_COFFEE17_PREPROCESSING_DATA_GATE':
    raise RuntimeError('Data gate gagal')

DATA_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-data-v1'
RUNTIME_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-runtime-v1'
STATIC_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-static-v2'
OBS_EVIDENCE=PROJECT/'evidence/coffee17-preprocessing-observability-v1'
SETUP_LOCKS=PROJECT/'locks/coffee17-preprocessing-v1'
for directory in (DATA_EVIDENCE,RUNTIME_EVIDENCE,STATIC_EVIDENCE,OBS_EVIDENCE,SETUP_LOCKS):
    directory.mkdir(parents=True,exist_ok=True)

def persist_exact(source,target):
    source=Path(source); target=Path(target)
    if target.is_file():
        if sha256_file(source)!=sha256_file(target):
            raise RuntimeError(f'Persistent Kaggle evidence berbeda: {target}')
    else:
        target.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(source,target)

with exclusive_training_lock(SETUP_LOCKS,lock_name='common_setup.lock',stale_seconds=900):
    persist_exact(PROV_LOCAL/'coffee17_provenance.json',DATA_EVIDENCE/'coffee17_provenance.json')
    persist_exact(PROV_LOCAL/'coffee17_raw_manifest.json',DATA_EVIDENCE/'coffee17_raw_manifest.json')
    for name in ('clean_manifest.json','fold_manifest.json','fold_summary.json'):
        persist_exact(FOLDS_LOCAL/name,DATA_EVIDENCE/name)

    ENV=RUNTIME_EVIDENCE/'runtime_environment.json'
    LOCK=RUNTIME_EVIDENCE/'requirements_preprocessing_study_lock.txt'
    if ENV.is_file():
        verify_environment(ENV)
    else:
        freeze_environment(ENV,LOCK,authorize_freeze=True)

    STATIC=STATIC_EVIDENCE/'static_preflight.json'
    if STATIC.is_file():
        static_existing=json.loads(STATIC.read_text())
        if static_existing.get('decision')!='PASS_PREPROCESSING_STATIC_CONTRACT':
            raise RuntimeError('Static evidence belum PASS')
    else:
        LOCAL_STATIC=WORK/'static_preflight.json'
        run_preprocessing_static_preflight(LOCAL_STATIC,seed=42,probe_size=64)
        persist_exact(LOCAL_STATIC,STATIC)

    OBS=OBS_EVIDENCE/'preprocessing_observability.json'
    if OBS.is_file():
        obs_existing=json.loads(OBS.read_text())
        if obs_existing.get('decision')!='PASS_PREPROCESSING_OBSERVABILITY_AUDIT':
            raise RuntimeError('Observability evidence belum PASS')
    else:
        LOCAL_OBS=WORK/'preprocessing_observability'
        if LOCAL_OBS.exists():
            shutil.rmtree(LOCAL_OBS)
        run_observability_audit(
            CANONICAL,
            PROV_LOCAL/'coffee17_raw_manifest.json',
            LOCAL_OBS,
            device_name='cuda:0',
            image_size=224,
            expected_count=979,
        )
        persist_exact(LOCAL_OBS/'preprocessing_observability.json',OBS)
        persist_exact(
            LOCAL_OBS/'preprocessing_observability_per_image.csv',
            OBS_EVIDENCE/'preprocessing_observability_per_image.csv'
        )

verify_environment(ENV)
OUT=PROJECT/'experiments/coffee17-preprocessing-primary-v1'
OUT.mkdir(parents=True,exist_ok=True)

print('ARM:',ARM)
print('CODE_COMMIT:',CODE_COMMIT)
print('CLEAN:',fold_summary['clean_count'])
print('PROJECT:',PROJECT)
print('OBSERVABILITY:',OBS)


In [ ]:
obs=json.loads(OBS.read_text())
print('DECISION:',obs['decision'])
print('GATES:')
print(json.dumps(obs['gates'],indent=2))
print('ARMS:')
print(json.dumps(obs['arms'],indent=2))
print('\nR0 pertama: kirim output cell ini ke ChatGPT sebelum menjalankan cell TRAINING.')


In [ ]:
DET=WORK/'coffee-bean-detection-reference'
if DET.exists():
    shutil.rmtree(DET)
subprocess.run([
    'git','clone','--quiet','https://github.com/ediprin/coffee-bean-detection.git',str(DET)
],check=True)
REFERENCE_COMMIT='6ef389c23932e44fe4135c32d471b3008b1cbf39'
subprocess.run(['git','-C',str(DET),'checkout','--quiet','--force',REFERENCE_COMMIT],check=True)
EQ_DIR=PROJECT/'evidence/coffee17-preprocessing-reference-v2'
EQ_DIR.mkdir(parents=True,exist_ok=True)
EQUIVALENCE=EQ_DIR/'F0_luminance_reference_equivalence.json'
LOCAL_EQ=WORK/'F0_luminance_reference_equivalence.json'
verify_reference_equivalence(
    'F0',DET,LOCAL_EQ,expected_reference_commit=REFERENCE_COMMIT
)
persist_exact(LOCAL_EQ,EQUIVALENCE)
print('F0 REFERENCE EQUIVALENCE:',EQUIVALENCE)


In [ ]:
for FOLD in range(1,6):
    DEV=WORK/f'coffee17_dev_fold_{FOLD}'
    if DEV.exists():
        shutil.rmtree(DEV)
    materialize_preprocessing_development(
        CANONICAL,
        FOLDS_LOCAL/'clean_manifest.json',
        FOLDS_LOCAL/'fold_manifest.json',
        DEV,
        fold=FOLD,
    )
    CONTRACT=DEV/'development_contract.json'
    LOG=OUT/f'{ARM}_fold{FOLD}_seed42_run.log'
    command=[
        sys.executable,'-u','-m','bilinear_lmmd.experiments.run_preprocessing_arm',
        '--arm',ARM,'--fold',str(FOLD),'--seed','42',
        '--data-root',str(DEV),
        '--development-contract',str(CONTRACT),
        '--static-preflight',str(STATIC),
        '--observability-audit',str(OBS),
        '--environment',str(ENV),
        '--output-root',str(OUT),
        '--required-commit',CODE_COMMIT,
        '--device','cuda:0',
        '--authorize-training',
    ]
    if EQUIVALENCE is not None:
        command += ['--equivalence',str(EQUIVALENCE)]
    print(f'START/RESUME {ARM} fold {FOLD}/5',flush=True)
    with LOG.open('a',encoding='utf-8') as stream:
        process=subprocess.Popen(
            command,cwd=REPO,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line,end='')
            stream.write(line)
            stream.flush()
        rc=process.wait()
    if rc:
        raise RuntimeError(f'{ARM} fold {FOLD} gagal: {rc}')
    result_path=OUT/'primary'/ARM/f'fold_{FOLD}'/'seed42'/'result.json'
    result=json.loads(result_path.read_text())
    print(
        f"{ARM} fold {FOLD}: "
        f"Macro-F1={result['metrics']['macro_f1']:.4f} | "
        f"Worst-F1={result['metrics']['worst_class_f1']:.4f}"
    )
    shutil.rmtree(DEV,ignore_errors=True)
print(f'{ARM}: 5 fold selesai. Outer test belum dibuka.')


In [ ]:
marker_dir=PROJECT/'kaggle_runs'
marker_dir.mkdir(parents=True,exist_ok=True)
marker={
    'arm':ARM,
    'seed':42,
    'folds_complete':5,
    'code_commit':CODE_COMMIT,
    'test_images_accessed':False,
    'project_root':'coffee17-preprocessing-project',
}
(marker_dir/f'{ARM}_seed42_complete.json').write_text(
    json.dumps(marker,indent=2)+'\n',encoding='utf-8'
)
for path in (REPO,PROV_LOCAL,CANONICAL,FOLDS_LOCAL):
    if Path(path).exists():
        shutil.rmtree(path,ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
if ARM=='F0' and 'DET' in globals() and DET.exists():
    shutil.rmtree(DET,ignore_errors=True)
print('SELESAI:',ARM)
print('Kaggle output yang harus disimpan:',PROJECT)
print('Klik Save Version agar coffee17-preprocessing-project menjadi Notebook Output.')
